<div style="text-align: justify; max-width: 90ch">

# Lecture 01a: Your laptop as an LLM server

**Status: Mandatory reading.**

In the Python course you trained models yourself: `fit`, `predict`, a metric. A **large language model (LLM)** is different:
someone else trained it for months on thousands of GPUs, and your job is to *run* it and *talk* to it. This notebook makes that concrete on your own laptop, with no account and no API key.

**What this notebook covers**

- Two new tools: `uv` (replaces venv + pip) and **Ollama** (a local model server).
- Your first call to a model from Python, and what the response tells you.
- **Tokens**: the unit a model reads, writes, and is paid in.
- The **context window**: how much text a model can see at once.
- **Temperature**: why the same question gives different answers.

Before you start, "homework zero" must be done: `uv sync`, Ollama installed, `ollama pull qwen3:1.7b` (see `instructions_guides/`).

</div>

In [ ]:
import ollama
import pandas as pd
from transformers import AutoTokenizer

<div style="text-align: justify; max-width: 90ch">

## 0. The configuration cell

Every notebook in this course starts with the same cell. You edit **one line**: `TIER`. The `cpu` tier uses `qwen3:1.7b`, a model small enough for any laptop;
the `gpu` tier uses `qwen3:8b`, a bigger sibling for owners of a GPU with at least 8 GB of VRAM. Which fits *your* machine?
`uvx llmfit recommend` tells you (guide 00d), and lecture 01b explains what else goes into that choice.

The last line calls a helper from `llm_course`, the small package installed by `uv sync`. It checks that Ollama answers and that the models are pulled, and if not it prints the exact command to run.

</div>

In [ ]:
# --- Course configuration: edit TIER only -------------------------------------
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

import os
from pathlib import Path

TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
EXPERIMENT = "llm-course-01-basics"
REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Two new tools

**`uv`** does what `python -m venv` plus `pip install -r requirements.txt` did in the Python course, in one command (`uv sync`),
and it records the exact versions in `uv.lock` so everyone runs the same code. Which interpreter a notebook uses depends on how you open it:
`uv run jupyter lab` runs Jupyter inside `.venv/` automatically; in VS Code you select the `.venv` kernel yourself (guide 00a, section 4).

```bash
# WRONG: installs into whatever Python is first on PATH, invisible to uv.lock
pip install ollama

# RIGHT: adds the package to pyproject.toml and uv.lock, then installs it
uv add ollama
```

**Ollama** is a system-native application (installed like a browser, not like a Python package) that runs open models on your computer and exposes them as a **local web server** on port `11434`.
The `ollama` Python package that `uv sync` installed is only a client for that server (guide 00b). That is the same idea as `mlflow server` in lecture 13:
a process that keeps running, and Python code that talks to it over HTTP. Ollama downloads models with `ollama pull <name>` and keeps them in a local folder, like `git` for models.

Let's ask the server what it has.

</div>

In [ ]:
client = ollama.Client(host=OLLAMA_HOST)

models = client.list().models
pd.DataFrame(
    {
        "model": [m.model for m in models],
        "size_gb": [round(m.size / 1e9, 2) for m in models],
        "parameters": [m.details.parameter_size for m in models],
        "quantization": [m.details.quantization_level for m in models],
    }
)

,model,size_gb,parameters,quantization
0,qwen3:0.6b,0.52,751.63M,Q4_K_M
1,hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0,0.64,596M,unknown
2,hf.co/Qwen/Qwen3-0.6B-GGUF:Q8_0,0.64,596M,unknown
3,qwen3:1.7b,1.36,2.0B,Q4_K_M
4,gemma3:4b,3.34,4.3B,Q4_K_M
5,nomic-embed-text:latest,0.27,137M,F16
6,qwen3:8b,5.23,8.2B,Q4_K_M


<div style="text-align: justify; max-width: 90ch">

Three columns deserve a first look:

- **parameters**: the number of weights, the "size" of the model. `1.7B` means 1.7 billion numbers.
- **quantization**: how many bits each weight is stored in. `Q4_K_M` is 4 bits, which is why 1.7 billion weights take 1.4 GB, not 3.4 GB.
  Lecture 01b returns to this.
- **size_gb**: what that costs on disk, and roughly in RAM when the model is loaded.

`nomic-embed-text` is not a chat model; lecture 2 uses it to turn text into numbers.

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Your first call

An LLM does one thing: given text, it predicts the next **token** (a word piece), appends it, and repeats until it emits a stop token.
Everything else, chat, coding, summarising, is that loop with a clever prompt.

You send a **list of messages**, each with a `role` and `content`. For now there is one message, from you, the `user`.

`qwen3:1.7b` is a **reasoning model**: by default it "thinks out loud" before answering, which is slow. We pass `think=False` to switch that off;
lecture 01b shows what thinking costs and when it is worth it.

</div>

In [ ]:
response = client.chat(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "In one sentence, what is data leakage in machine learning?",
        }
    ],
    think=False,
    options={
        "temperature": 0,
        "num_predict": 150,
    },  # num_predict caps the answer length
)
print(response.message.content)

Data leakage in machine learning refers to a situation where a model inadvertently learns relationships between the training data and the test data that are not intended or valid, leading to overly optimistic performance estimates.


<div style="text-align: justify; max-width: 90ch">

The response carries more than the text. Ollama reports how many tokens it read, how many it wrote, and how long each took, in nanoseconds.

</div>

In [ ]:
tokens_in = response.prompt_eval_count
tokens_out = response.eval_count
seconds_out = response.eval_duration / 1e9

print(f"prompt tokens : {tokens_in}")
print(f"answer tokens : {tokens_out}")
print(
    f"generation    : {seconds_out:.1f} s  ->  {tokens_out / seconds_out:.1f} tokens/s"
)
print(
    f"total         : {response.total_duration / 1e9:.1f} s (includes loading the model the first time)"
)

prompt tokens : 28
answer tokens : 38
generation    : 0.4 s  ->  108.3 tokens/s
total         : 1.5 s (includes loading the model the first time)


<div style="text-align: justify; max-width: 90ch">

Tokens per second is the number to remember when you compare models or machines. In one example run on a laptop CPU, `qwen3:1.7b` writes 10-25 tokens/s;
on a GPU, over 100.

### ⏸ Pause point

Run `client.ps()` below and compare with your neighbours: is the model on the CPU or the GPU, and how much memory does it use?

</div>

In [ ]:
loaded = client.ps().models
pd.DataFrame(
    {
        "model": [m.model for m in loaded],
        "memory_gb": [round(m.size / 1e9, 2) for m in loaded],
        "on_gpu_gb": [round(m.size_vram / 1e9, 2) for m in loaded],
    }
)

,model,memory_gb,on_gpu_gb
0,qwen3:1.7b,1.90,1.90
1,hf.co/unsloth/Qwen3-0.6B-GGUF:Q8_0,1.32,1.32


<div style="text-align: justify; max-width: 90ch">

`on_gpu_gb` equal to `memory_gb` means the whole model sits on a GPU; `0.0` means it runs on the CPU. Ollama keeps a model loaded for five minutes after the last call, so the second call is faster than the first.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Tokens

A **token** is the unit of text a model works with: a word, a word piece, or a punctuation mark, chosen by a **tokenizer** that was fixed before training.
Tokens matter three times over:

- **Cost**: hosted models charge per token; local models spend time per token.
- **Capacity**: the context window (next section) is measured in tokens.
- **Speed**: generation is one token at a time.

The tokenizer is a small file published with the model on the Hugging Face Hub. The `transformers` library downloads it (about 10 MB, one-time) so we can count tokens without running the model.

</div>

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")

english = "Data leakage happens when the test set influences training."
ids = tokenizer.encode(english)
print(len(english), "characters ->", len(ids), "tokens")
print(tokenizer.convert_ids_to_tokens(ids))

59 characters -> 10 tokens
['Data', 'Ġleakage', 'Ġhappens', 'Ġwhen', 'Ġthe', 'Ġtest', 'Ġset', 'Ġinfluences', 'Ġtraining', '.']


<div style="text-align: justify; max-width: 90ch">

`Ġ` marks a token that starts with a space: `Ġleakage` is " leakage". Ten tokens for 59 characters, about 4 characters per token.
That rule of thumb is for English. Watch what happens to the same sentence in Greek.

</div>

In [ ]:
greek = "Διαρροή δεδομένων συμβαίνει όταν το test set επηρεάζει την εκπαίδευση."

rows = []
for label, text in [("English", english), ("Greek", greek)]:
    n_tokens = len(tokenizer.encode(text))
    rows.append(
        {
            "language": label,
            "characters": len(text),
            "tokens": n_tokens,
            "chars_per_token": round(len(text) / n_tokens, 1),
        }
    )
pd.DataFrame(rows)

,language,characters,tokens,chars_per_token
0,English,59,10,5.9
1,Greek,70,57,1.2


<div style="text-align: justify; max-width: 90ch">

Greek costs about five times more tokens for the same sentence, because the tokenizer's vocabulary was built mostly from English text and has to spell Greek words letter by letter.
Two consequences you will meet again:

- A Greek prompt fills the context window five times faster and generates five times slower.
- Which model to pick for a Greek use case is partly a **tokenizer** question (lecture 01b).

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. The context window

The **context window** is the maximum number of tokens a model can see at once: your messages plus its answer. Text beyond it is simply not there for the model.
Ollama lets you set the window per call with `num_ctx`; a bigger window costs more RAM.

Let's put a fact at the *start* of a long text and ask about it. First we make sure the text is longer than the window we will use.

</div>

In [ ]:
needle = "The secret word for this lecture is PELICAN."
filler = " ".join(
    f"Note {i}: in the Python course we used pandas for tables, seaborn for plots, and scikit-learn for models."
    for i in range(60)
)
long_text = needle + " " + filler
question = "What is the secret word mentioned in the text? Answer with the word only."

print("tokens in the text:", len(tokenizer.encode(long_text)))

tokens in the text: 1622


In [ ]:
# WRONG: a 512-token window; the start of the text is cut off, the model never sees the needle
response_small = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 512, "num_predict": 20},
)
print("num_ctx=512  ->", response_small.message.content.strip())

num_ctx=512  -> models


In [ ]:
# RIGHT: a window large enough for the whole text
response_large = client.chat(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": long_text + "\n\n" + question}],
    think=False,
    options={"temperature": 0, "num_ctx": 4096, "num_predict": 20},
)
print("num_ctx=4096 ->", response_large.message.content.strip())

num_ctx=4096 -> PELICAN


<div style="text-align: justify; max-width: 90ch">

With the small window the model answers about the text it *can* see, or guesses. Nothing warned you: overflow is silent, which is why lecture 2 counts tokens before sending documents to a model.

`qwen3:1.7b` supports up to about 40 000 tokens; Ollama's default window is smaller (4096 in recent versions) to save memory.
`ollama show qwen3:1.7b` in a terminal prints both.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. Temperature

The model does not output one token; it outputs a probability for *every* token in its vocabulary. **Temperature** controls how that distribution is sampled:
at `0` the most likely token is picked every time; at `1` less likely tokens get their chance.

Same prompt, three runs at each setting.

</div>

In [ ]:
prompt = "Suggest a name for a coffee shop run by data scientists. Answer with the name only."

rows = []
for temperature in [0.0, 1.0]:
    for run in range(3):
        answer = client.chat(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            think=False,
            options={"temperature": temperature, "num_predict": 20},
        ).message.content.strip()
        rows.append({"temperature": temperature, "run": run + 1, "answer": answer})
pd.DataFrame(rows)

,temperature,run,answer
0,0.0,1,DataBrew
1,0.0,2,DataBrew
2,0.0,3,DataBrew
3,1.0,1,DataCafé
4,1.0,2,DataBrew
5,1.0,3,NeuroCafe


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Which setting would you ship in a customer-facing product, and which in a brainstorming tool? In this course we use `temperature=0` almost everywhere, for the same reason you set `random_state=42` in scikit-learn:
when a number changes, we want to know it was our change, not luck. (Temperature 0 is *nearly* deterministic; on some hardware two runs can still differ.)

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- `uv sync` builds the environment from `uv.lock`; `uv run` runs tools inside it; never `pip install`.
- Ollama is a **local model server**; `ollama pull` fetches models, `client.chat` talks to them, `client.ps()` shows where they run.
- An LLM predicts the next **token**; the response reports tokens in, tokens out, and tokens per second.
- The tokenizer decides the cost: Greek text is several times more expensive than English.
- The **context window** is a hard, silent limit measured in tokens; `num_ctx` sets it per call.
- **Temperature** trades reproducibility for variety; the course uses `0`.

**Next: `lec_01b`**, choosing a model. Size is only one of the things that decide which model you run; the requirements of the job decide the rest.

</div>